In [1]:

import h5py
with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    def print_all(name, obj):
        if not name.startswith('feature_slices'):
            print(name)
    f.visititems(print_all)
    # top level keys
    print('TOP LEVEL:', list(f.keys()))


features
features/_all_tag_keys
features/feature_type
features/genome
features/id
features/name
features/target_sets
features/target_sets/Visium Human Transcriptome Probe Set v2.0
images
images/cytassist
images/cytassist/col
images/cytassist/data
images/cytassist/row
images/microscope
images/microscope/col
images/microscope/data
images/microscope/row
masks
masks/filtered
masks/filtered/col
masks/filtered/data
masks/filtered/row
masks/square_008um
masks/square_008um/col
masks/square_008um/data
masks/square_008um/row
masks/square_016um
masks/square_016um/col
masks/square_016um/data
masks/square_016um/row
reads
reads/barcode_corrected_sequenced
reads/barcode_corrected_sequenced/col
reads/barcode_corrected_sequenced/data
reads/barcode_corrected_sequenced/row
reads/half_mapped
reads/half_mapped/col
reads/half_mapped/data
reads/half_mapped/row
reads/sequenced
reads/sequenced/col
reads/sequenced/data
reads/sequenced/row
reads/split_mapped
reads/split_mapped/col
reads/split_mapped/data
reads/s

In [2]:

import h5py
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # 마스크 확인
    for key in ['masks/filtered', 'masks/square_008um', 'masks/square_016um']:
        col = f[key+'/col'][:]
        row = f[key+'/row'][:]
        data = f[key+'/data'][:]
        print(f'{key}: {len(data)} entries, data range: {data.min()}-{data.max()}')
        print(f'  row range: {row.min()}-{row.max()}, col range: {col.min()}-{col.max()}')

    # 이미지 확인
    for key in ['images/cytassist', 'images/microscope']:
        col = f[key+'/col'][:]
        row = f[key+'/row'][:]
        data = f[key+'/data'][:]
        print(f'{key}: data shape {data.shape}, row range {row.min()}-{row.max()}, col range {col.min()}-{col.max()}')


masks/filtered: 8812974 entries, data range: 1-1
  row range: 0-3349, col range: 0-3349
masks/square_008um: 553820 entries, data range: 1-1
  row range: 0-837, col range: 0-837
masks/square_016um: 139406 entries, data range: 1-1
  row range: 0-418, col range: 0-418
images/cytassist: data shape (11222499,), row range 0-3349, col range 0-3349
images/microscope: data shape (10998169,), row range 0-3349, col range 62-3349


In [3]:

import h5py
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # 이미지가 RGB인지, 픽셀값이 뭔지 확인
    data = f['images/microscope/data'][:]
    print('data dtype:', data.dtype)
    print('data shape:', data.shape)
    print('data sample (first 10):', data[:10])
    
    # 전체 이미지 크기 확인
    row = f['images/microscope/row'][:]
    col = f['images/microscope/col'][:]
    print('unique rows:', len(np.unique(row)))
    print('unique cols:', len(np.unique(col)))
    print('total grid size:', row.max()+1, 'x', col.max()+1)


data dtype: uint8
data shape: (10998169,)
data sample (first 10): [  4 244 243 243 244 243 243 243 243 244]
unique rows: 3350
unique cols: 3288
total grid size: 3350 x 3350


In [4]:

import h5py
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # images 그룹 속성 확인
    img_grp = f['images/microscope']
    print('attributes:', dict(img_grp.attrs))
    print('keys:', list(img_grp.keys()))
    
    # data dataset 속성 확인
    d = img_grp['data']
    print('data attrs:', dict(d.attrs))
    
    # spatial 이미지 확인 (PNG 파일)
    from PIL import Image
    img = Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png')
    print('hires image size:', img.size)
    print('hires image mode:', img.mode)
    
    img2 = Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/aligned_tissue_image.jpg')
    print('aligned image size:', img2.size)


attributes: {'metadata_json': '{"kind": "gray_image_u8", "domain": "all_spots", "binning_scale": 1}'}
keys: ['col', 'data', 'row']
data attrs: {}
hires image size: (5259, 6000)
hires image mode: RGB
aligned image size: (5259, 6000)


In [5]:

import h5py
import numpy as np
from PIL import Image

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # scalefactors 같은 메타데이터 있는지 확인
    print('top level attrs:', dict(f.attrs))
    
    # masks/filtered 실제로 재구성해보기
    row = f['masks/filtered/row'][:]
    col = f['masks/filtered/col'][:]
    data = f['masks/filtered/data'][:]
    
    mask = np.zeros((3350, 3350), dtype=np.uint8)
    mask[row, col] = data
    print('mask shape:', mask.shape)
    print('tissue pixels:', mask.sum())
    print('total pixels:', mask.size)
    print('tissue ratio:', mask.sum()/mask.size)


top level attrs: {'filetype': 'feature_slice', 'metadata_json': '{"sample_id": "Visium_HD_FF_Human_Tonsil", "sample_desc": "Gene expression library of Fresh Frozen Human Tonsil (Visium HD) using the Human Whole Transcriptome Probe Set", "slide_name": "visium_hd_v1", "nrows": 3350, "ncols": 3350, "spot_pitch": 2.0, "hd_layout_json": "{\\"slide_uid\\":\\"H1-BHM9GT7\\",\\"file_format\\":\\"1.0\\",\\"aligner_version\\":\\"1.0\\",\\"input_hash\\":\\"96bde14d95ba0743d7e9562b89160cac\\",\\"slide_design\\":\\"visium_hd_rc1\\",\\"transform\\":[1.0000325399274357,0.002591775464519526,-33.657701753274374,-0.0025907969752494314,1.0000102123948542,63.96893850375202,9.414438094981444e-9,-2.285391216086935e-9,1.0]}", "transform_matrices": {"spot_colrow_to_microscope_colrow": [[-3.65394524181981, 0.002133389590472481, 15524.64179357808], [0.0030534266764736047, 3.6526930255968555, 2020.7158251302426], [1.8743537078861507e-08, -4.7170217319431605e-09, 1.0]], "microscope_colrow_to_spot_colrow": [[-0.273

In [6]:

import json, h5py
with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
    print('spot_pitch:', meta['spot_pitch'], 'µm')  # bin 하나 크기
    print('nrows:', meta['nrows'], 'ncols:', meta['ncols'])  # spot grid 크기

from PIL import Image
img = Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png')
print('image size (px):', img.size)  # (width, height)


spot_pitch: 2.0 µm
nrows: 3350 ncols: 3350
image size (px): (5259, 6000)


In [7]:

import json
import h5py
from PIL import Image

# Visium HD 메타데이터
with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
    print('spot_pitch:', meta['spot_pitch'], 'um')
    print('nrows:', meta['nrows'], 'ncols:', meta['ncols'])

img = Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png')
print('hires image size (px):', img.size)

# Visium 일반 샘플 scalefactors 확인 (GSE252265 중 하나)
import glob
sf_files = glob.glob('/project_antwerp/hbae/data/**/scalefactors_json.json', recursive=True)
for sf in sf_files[:3]:
    with open(sf) as f:
        sf_data = json.load(f)
    print(sf, sf_data)


spot_pitch: 2.0 um
nrows: 3350 ncols: 3350
hires image size (px): (5259, 6000)


In [ ]:
find /project_antwerp/hbae/data/visium_hd_tonsil/ -name "scalefactors*" 2>/dev/null

# 없으면 spatial 폴더 전체 확인
ls -la /project_antwerp/hbae/data/visium_hd_tonsil/spatial/